# Part 6: The Trainer class

The `Trainer` class runs the loop for you: batching, the optimizer, the learning-rate schedule, evaluation and logging. It is the Hugging Face counterpart of Keras `fit`. This part runs the same fine-tuning as Part 5.

In [ ]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from review_data import as_lists, load_reviews, small_sample

reviews = load_reviews()
val_texts, val_labels = as_lists(reviews["validation"])

texts, labels = small_sample(reviews["train"], 200, seed=42)

In [ ]:
from trainer_version import fine_tune_with_trainer

trainer, tokenizer = fine_tune_with_trainer(texts, labels, val_texts, val_labels)

In [ ]:
for entry in trainer.state.log_history:
    if "eval_accuracy" in entry:
        print(round(entry["epoch"]), round(entry["eval_loss"], 4),
              round(entry["eval_accuracy"], 4))

## How the two versions line up

| Your loop in Part 5 | The Trainer |
|---|---|
| `DataLoader` and `DataCollatorWithPadding` | `data_collator=DataCollatorWithPadding(tokenizer)` |
| `AdamW` and the linear schedule | `learning_rate`, `weight_decay`, `warmup_steps` in `TrainingArguments` |
| `loss.backward()`, `optimizer.step()` | inside `trainer.train()` |
| `predict` and `scores` | `compute_metrics` |
| the history list | `trainer.state.log_history` |

The numbers differ a little from Part 5 because the defaults differ, for example in how the warm-up is counted. That is normal. What matters is that you can say what each setting does.

## Try it

1. Change `eval_strategy` so that evaluation runs every 5 steps. What do you see?
2. Set `num_train_epochs` to 1. How far does the accuracy fall?
3. Find where the Trainer stores the learning rate it used at each logged step.